# Prova half-width: quando si aprono i canali durante lo spike?

**Domanda:** nel modello lo spike è troppo largo (half-width circa 0.7 ms contro 0.22–0.31 ms nei neuroni veri,
Markkula e Choudhury). L'ipotesi è che **Kv3 si apra troppo tardi**, cioè quando lo spike è già quasi finito,
invece che durante la discesa.

**Input:** 5 impulsi di corrente uguali, da **1.4 nA** per **0.25 ms** (gli stessi dei treni di Macica, Fig. 9),
a **100 Hz** (uno ogni 10 ms). A 100 Hz ogni spike ha tempo di finire prima del successivo, quindi si vedono
5 spike "puliti" uno dopo l'altro.

**Cosa si registra:**
- il voltaggio V_m;
- i **cancelli** di ogni canale, da cui si calcola la **frazione di canali aperti**:
  - sodio: m³·h
  - Kv3: n³·(0.8 + 0.2 p)
  - I_LT (Kv1): l·r
- le **correnti** I_Na, I_Kv3, I_LT, I_L.

**Cosa ci si aspetta in un neurone che funziona bene:** il sodio si apre sulla salita, Kv3 si apre subito dopo
e raggiunge il massimo **durante la discesa**, così la discesa è rapida e lo spike stretto.

Il modello si sceglie con `MODEL_FILE`: funziona sia con `mntb_neuron_KV3.1_KV3.3_prova.nestml`
(Kv3 = Kv3.1 + Kv3.3) sia con i modelli con un solo Kv3 (`mntb_hh_kv3_neuron*.nestml`).

In [ ]:
import os
os.environ["PYNEST_QUIET"] = "1"
import numpy as np
import matplotlib.pyplot as plt
import nest
nest.set_verbosity("M_ERROR")
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

MODEL_FILE = "mntb_neuron_KV3.1_KV3.3_prova.nestml"
hw_module, hw_model = NESTCodeGeneratorUtils.generate_code_for(MODEL_FILE)

dt = 0.01                     # ms
AMP, DUR = 1400.0, 0.25       # pA, ms: impulso di corrente
FREQ, N_PULSES = 100.0, 5     # Hz, numero di impulsi
T_START = 20.0                # ms, primo impulso

nest.ResetKernel()
nest.Install(hw_module)
P = nest.GetDefaults(hw_model)
SPLIT = "g_Kv3_1" in P        # True se il modello ha Kv3.1 e Kv3.3 separati
print(f"Modello: {hw_model}  (Kv3 diviso in Kv3.1/Kv3.3: {SPLIT})")

# stimolo
per = 1000.0 / FREQ
times, values, onsets = [], [], []
for i in range(N_PULSES):
    t0 = round((T_START + i * per) / dt) * dt
    times += [t0, round((t0 + DUR) / dt) * dt]
    values += [AMP, 0.0]
    onsets.append(t0 + dt)          # lo stimolo arriva dopo il ritardo di connessione (dt)

gates = ["Act_m", "Inact_h", "Act_l", "Inact_r"] + (["Act_n1", "Inact_p1", "Act_n3", "Inact_p3"] if SPLIT else ["Act_n", "Inact_p"])

nest.SetKernelStatus({"resolution": dt})
neuron = nest.Create(hw_model)
stim = nest.Create("step_current_generator", params={"amplitude_times": times, "amplitude_values": values})
mm = nest.Create("multimeter", params={"record_from": ["V_m"] + gates, "interval": dt})
nest.Connect(stim, neuron, syn_spec={"delay": dt})
nest.Connect(mm, neuron)
nest.Simulate(T_START + N_PULSES * per + 5.0)

ev = mm.get("events")
o = np.argsort(ev["times"])
t = ev["times"][o]
R = {k: ev[k][o] for k in ["V_m"] + gates}
V = R["V_m"]

# frazione di canali aperti (tra 0 e 1) e correnti (pA; + entrante, - uscente)
open_Na = R["Act_m"]**3 * R["Inact_h"]
open_LT = R["Act_l"] * R["Inact_r"]
if SPLIT:
    g_Kv3 = P["g_Kv3_1"] + P["g_Kv3_3"]
    open_Kv3 = (P["g_Kv3_1"] * R["Act_n1"]**3 * (0.8 + 0.2 * R["Inact_p1"])
                + P["g_Kv3_3"] * R["Act_n3"]**3 * (0.8 + 0.2 * R["Inact_p3"])) / g_Kv3
else:
    g_Kv3 = P["g_Kv3"]
    open_Kv3 = R["Act_n"]**3 * (0.8 + 0.2 * R["Inact_p"])
I_Na = P["g_Na"] * open_Na * (P["E_Na"] - V)
I_Kv3 = g_Kv3 * open_Kv3 * (P["E_K"] - V)
I_LT = P["g_LT"] * open_LT * (P["E_K"] - V)
I_L = P["g_L"] * (P["E_L"] - V)
print(f"g_Na = {P['g_Na']:g} nS, g_Kv3 = {g_Kv3:g} nS, g_LT = {P['g_LT']:g} nS, g_L = {P['g_L']:g} nS")

## Misure per ogni spike

Per ogni spike si misurano:
- **inizio e fine dello spike** = istanti in cui V passa la metà dell'ampiezza (salendo e scendendo):
  la distanza tra i due è la half-width;
- l'istante del **massimo** di V, del sodio aperto, di Kv3 aperto e di I_LT aperta;
- **quanto Kv3 è aperto al picco e alla fine dello spike**, in percentuale del suo massimo.

Se Kv3 raggiunge il massimo **dopo la fine dello spike**, si apre troppo tardi per accorciarlo.

In [ ]:
rows = []
for k, t0 in enumerate(onsets):
    w = (t >= t0) & (t < t0 + per)
    idx = np.where(w)[0]
    rest = V[(t >= t0 - 2.0) & (t < t0)].mean()
    ip = idx[np.argmax(V[idx])]
    half = rest + (V[ip] - rest) / 2
    i1 = idx[0] + np.argmax(V[idx[0]:ip + 1] >= half)
    i2 = ip + np.argmax(V[ip:] <= half)
    iNa = idx[np.argmax(open_Na[idx])]
    iK = idx[np.argmax(open_Kv3[idx])]
    iLT = idx[np.argmax(open_LT[idx])]
    rows.append(dict(spike=k + 1, t_on=t[i1], t_peak=t[ip], t_off=t[i2], hw=t[i2] - t[i1],
                     t_Na=t[iNa], t_Kv3=t[iK], t_LT=t[iLT], V_at_Kv3max=V[iK],
                     kv3_at_peak=open_Kv3[ip] / open_Kv3[iK], kv3_at_end=open_Kv3[i2] / open_Kv3[iK],
                     kv3_max=open_Kv3[iK], na_max=open_Na[iNa]))

print("Tempi misurati dal picco dello spike (ms); + = dopo il picco")
print(f"{'spike':>5}{'half-width':>11}{'fine spike':>11}{'max Na':>8}{'max Kv3':>9}{'max I_LT':>9}"
      f"{'V a max Kv3':>12}{'Kv3 al picco':>13}{'Kv3 a fine spike':>17}")
for r in rows:
    print(f"{r['spike']:>5}{r['hw']:>11.3f}{r['t_off'] - r['t_peak']:>+11.3f}{r['t_Na'] - r['t_peak']:>+8.3f}"
          f"{r['t_Kv3'] - r['t_peak']:>+9.3f}{r['t_LT'] - r['t_peak']:>+9.3f}{r['V_at_Kv3max']:>10.1f} mV"
          f"{100 * r['kv3_at_peak']:>11.0f} %{100 * r['kv3_at_end']:>15.0f} %")
r = rows[2]
print()
print(f"Spike 3: il sodio e' al massimo {r['t_Na'] - r['t_peak']:+.2f} ms dal picco, lo spike finisce a {r['t_off'] - r['t_peak']:+.2f} ms,")
print(f"Kv3 raggiunge il massimo a {r['t_Kv3'] - r['t_peak']:+.2f} ms, quando V e' gia' a {r['V_at_Kv3max']:.1f} mV.")
print(f"Al picco dello spike Kv3 e' aperto solo al {100 * r['kv3_at_peak']:.0f}% del suo massimo; a fine spike al {100 * r['kv3_at_end']:.0f}%.")
print(f"Frazione massima di canali aperti: sodio {r['na_max']:.2f}, Kv3 {r['kv3_max']:.3f}")

## Figura 1 – I 5 spike e l'apertura dei canali

- **A:** voltaggio; le bande grigie sono la durata degli spike (half-width), le linee verticali gli impulsi.
- **B:** frazione di canali aperti, ognuna **normalizzata al suo massimo** (0 = chiusi, 1 = massimo raggiunto),
  per confrontare i **tempi** di apertura anche se i valori assoluti sono molto diversi.
- **C:** correnti (+ entrante = depolarizza, − uscente = ripolarizza).

In [ ]:
c_Na, c_K, c_LT, c_L = "tab:orange", "tab:blue", "tab:purple", "grey"
fig, ax = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for r in rows:
    for a in ax:
        a.axvspan(r["t_on"], r["t_off"], color="lightgrey", alpha=0.6, lw=0)
for t0 in onsets:
    for a in ax:
        a.axvline(t0, color="k", ls=":", lw=0.6)

ax[0].plot(t, V, "k", lw=1.2)
ax[0].set_ylabel("V (mV)")
ax[0].set_title("A. Cinque spike a 100 Hz (impulsi 1.4 nA, 0.25 ms; banda grigia = durata dello spike a metà ampiezza)")

ax[1].plot(t, open_Na / open_Na.max(), color=c_Na, label="sodio aperto  m³h")
ax[1].plot(t, open_Kv3 / open_Kv3.max(), color=c_K, lw=2, label="Kv3 aperto  n³(0.8+0.2p)")
ax[1].plot(t, open_LT / open_LT.max(), color=c_LT, label="I_LT aperta  l·r")
ax[1].set_ylabel("frazione aperta\n(normalizzata al max)")
ax[1].set_title("B. Quando si aprono i canali")
ax[1].legend(loc="upper right", fontsize=9)

ax[2].plot(t, I_Na / 1000, color=c_Na, label="I_Na")
ax[2].plot(t, I_Kv3 / 1000, color=c_K, lw=2, label="I_Kv3")
ax[2].plot(t, I_LT / 1000, color=c_LT, label="I_LT")
ax[2].plot(t, I_L / 1000, color=c_L, label="I_L")
ax[2].set_ylim(-3, 3)
ax[2].set_ylabel("I (nA)\n+ entrante / − uscente")
ax[2].set_xlabel("t (ms)")
ax[2].set_title("C. Correnti (I_Na esce dal grafico in alto: arriva a decine di nA)")
ax[2].legend(loc="upper right", fontsize=9)
for a in ax:
    a.grid(alpha=0.3)
ax[0].set_xlim(onsets[0] - 2, onsets[-1] + per - 2)
plt.tight_layout(); plt.show()

## Figura 2 – Zoom sul terzo spike

Stesse curve della Figura 1 ma su un solo spike, con il tempo misurato **dal picco** dello spike.
Le linee verticali indicano: picco dello spike (nero), fine dello spike (grigio), massimo del sodio aperto
(arancione) e massimo di Kv3 aperto (blu).

In [ ]:
r = rows[2]
tp = r["t_peak"]
w = (t >= tp - 1.0) & (t <= tp + 6.0)
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))

a = ax[0]
a.axvspan(r["t_on"] - tp, r["t_off"] - tp, color="lightgrey", alpha=0.6, lw=0, label="durata dello spike")
a.plot(t[w] - tp, (V[w] - V[w].min()) / (V[w].max() - V[w].min()), "k", lw=2, label="V (normalizzato)")
a.plot(t[w] - tp, open_Na[w] / open_Na[w].max(), color=c_Na, label="sodio aperto")
a.plot(t[w] - tp, open_Kv3[w] / open_Kv3[w].max(), color=c_K, lw=2, label="Kv3 aperto")
a.plot(t[w] - tp, open_LT[w] / open_LT[w].max(), color=c_LT, label="I_LT aperta")
for tt, col in ((0.0, "k"), (r["t_off"] - tp, "grey"), (r["t_Na"] - tp, c_Na), (r["t_Kv3"] - tp, c_K)):
    a.axvline(tt, color=col, ls="--", lw=1)
a.text(1.3, 1.02,
       f"Fine dello spike: {r['t_off'] - tp:+.2f} ms dal picco\n"
       f"Kv3 al massimo: {r['t_Kv3'] - tp:+.2f} ms (V = {r['V_at_Kv3max']:.0f} mV)\n"
       f"Kv3 al picco dello spike: {100 * r['kv3_at_peak']:.0f}% del suo massimo\n"
       f"Kv3 aperto al massimo: {100 * r['kv3_max']:.1f}% dei canali (sodio: {100 * r['na_max']:.0f}%)",
       color=c_K, fontsize=9, va="bottom", bbox=dict(facecolor="white", edgecolor=c_K, alpha=0.9))
a.set_ylim(-0.05, 1.35)
a.set_xlabel("t dal picco dello spike (ms)"); a.set_ylabel("normalizzato al massimo")
a.set_title("Spike 3: tempi di apertura (normalizzati)")
a.legend(loc="center right", fontsize=8); a.grid(alpha=0.3)

a = ax[1]
a.axvspan(r["t_on"] - tp, r["t_off"] - tp, color="lightgrey", alpha=0.6, lw=0)
a.plot(t[w] - tp, I_Na[w] / 1000, color=c_Na, label="I_Na")
a.plot(t[w] - tp, I_Kv3[w] / 1000, color=c_K, lw=2, label="I_Kv3")
a.plot(t[w] - tp, I_LT[w] / 1000, color=c_LT, label="I_LT")
a.plot(t[w] - tp, I_L[w] / 1000, color=c_L, label="I_L")
a.set_ylim(-3, 3)
a2 = a.twinx()
a2.plot(t[w] - tp, V[w], "k:", lw=1)
a2.set_ylabel("V (mV), linea puntinata")
a.set_xlabel("t dal picco dello spike (ms)"); a.set_ylabel("I (nA)   + entrante / − uscente")
a.set_title("Spike 3: correnti (valori veri)")
a.legend(loc="lower right", fontsize=8); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Figura 3 – Perché: la velocità dei cancelli

Ogni cancello x si muove verso il suo valore di equilibrio x∞(V) con una costante di tempo τ(V):
se τ è più lunga dello spike, il cancello **non fa in tempo** a seguirlo.

Si confrontano le τ dei cancelli con la durata di uno spike vero (half-width circa 0.22–0.31 ms) e con la
half-width del modello. A sinistra le curve x∞(V): a che tensione ogni cancello si apre.

In [ ]:
def rate(g, Vv):
    a = P["k_alpha_" + g] * np.exp(P["eta_alpha_" + g] * Vv)
    b = P["k_beta_" + g] * np.exp(P["eta_beta_" + g] * Vv)
    return a, b

n_name = "n1" if SPLIT else "n"
labels = {"m": ("m (sodio, apertura)", c_Na, "-"), "h": ("h (sodio, inattivazione)", c_Na, "--"),
          n_name: ("n (Kv3, apertura)", c_K, "-"), "l": ("l (I_LT, apertura)", c_LT, "-")}
Vv = np.linspace(-80, 50, 300)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for g, (lab, col, ls) in labels.items():
    a_, b_ = rate(g, Vv)
    ax[0].plot(Vv, a_ / (a_ + b_), color=col, ls=ls, lw=2 if g == n_name else 1.2, label=lab)
    ax[1].plot(Vv, 1.0 / (a_ + b_), color=col, ls=ls, lw=2 if g == n_name else 1.2, label=lab)
ax[0].set_xlabel("V (mV)"); ax[0].set_ylabel("x∞ (frazione all'equilibrio)")
ax[0].set_title("Cancelli all'equilibrio x∞(V)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].axhspan(0.22, 0.31, color="green", alpha=0.2, label="half-width neuroni veri (0.22–0.31 ms)")
ax[1].axhline(rows[2]["hw"], color="k", ls="--", lw=1, label=f"half-width del modello ({rows[2]['hw']:.2f} ms)")
ax[1].set_yscale("log")
ax[1].set_xlabel("V (mV)"); ax[1].set_ylabel("τ (ms), scala logaritmica")
ax[1].set_title("Costanti di tempo τ(V) = 1/(α+β)"); ax[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
ax[1].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

for Vx in (0.0, 20.0, 40.0):
    a_, b_ = rate(n_name, Vx)
    am, bm = rate("m", Vx)
    print(f"V = {Vx:+.0f} mV:  tau_n (Kv3) = {1 / (a_ + b_):.2f} ms,  n_inf = {a_ / (a_ + b_):.2f}   |   tau_m (sodio) = {1 / (am + bm):.3f} ms")

## Come leggere i risultati

- Se nella **Figura 2** la curva blu (Kv3) raggiunge il massimo **alla fine o dopo** la banda grigia, Kv3 si apre
  quando lo spike è già (quasi) finito: contribuisce poco ad accorciarlo e serve soprattutto all'iperpolarizzazione
  dopo lo spike. Conta anche **quanto** si apre: la percentuale di canali Kv3 aperti al massimo è scritta nel riquadro.
- Nella **Figura 3** il motivo: se τ_n (Kv3) a 0…+40 mV è di circa 1 ms o più, cioè più lunga dello spike,
  n non fa in tempo a salire durante lo spike. In più Kv3 dipende da **n³**: se n arriva solo a metà,
  n³ è appena 1/8 del massimo.
- **Due modi per correggere** (da discutere):
  1. **Kv3 più veloce:** moltiplicare α_n e β_n per lo stesso fattore F (n∞ non cambia, τ_n diventa F volte
     più corta). Giustificabile con la temperatura: le cinetiche sono state misurate a temperatura ambiente,
     e i canali a 35 °C sono 2–3 volte più veloci (Q10 ≈ 3).
  2. **Più Kv3:** aumentare g_Kv3. Choudhury misura circa 12 nA a +10 mV in cellule della stessa taglia
     del modello, che invece ne ha circa 5 (vedi `exp_KV3.1_KV3.3`).